# Notebook 6 — Step 7 Analyze performance
| | |
|---|---|
| **Case Study** | 6 — Smart City IoT Data Platform (6.1, 6.3 Real-time pipelines, ML systems) |
| **Roll No** | `<ROLL_NO>` |
| **Name** | `<NAME>` |

Three things are analysed:
1. **Batch optimizations** (Notebook 3): which optimization helped, by how much, and why.
2. **Streaming performance**: throughput, micro-batch duration and latency, compared with the real-time requirements.
3. **Model quality**: compared with the naive baselines.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
from src import config

## 1. Batch optimization results

In [ ]:
bench = pd.read_csv(config.PERF / "benchmark_results.csv")
meta = json.loads((config.PERF / "benchmark_results.json").read_text())
print(f"{meta['rows']:,} rows | {meta['cores']} cores | Spark {meta['spark']}")
bench[["experiment", "variant", "median_s", "speedup_vs_slowest_in_experiment", "note"]]

In [ ]:
summary = (bench.groupby("experiment")
           .apply(lambda d: pd.Series({"baseline_s": d["median_s"].iloc[0], "best_s": d["median_s"].min(),
                                       "best_variant": d.loc[d["median_s"].idxmin(), "variant"]}), include_groups=False))
summary["gain_pct"] = ((1 - summary["best_s"] / summary["baseline_s"]) * 100).round(1)
summary

**Interpretation guide** (fill in with your own numbers):
* **E1 caching** gains grow with the number of actions that reuse the DataFrame; with one action, caching only adds cost.
* **E2:** 200 shuffle partitions → 200 tasks of a few KB each; scheduling overhead dominates on a 4–8 core laptop.
* **E3:** a broadcast join removes the shuffle of the 4.8M-row side (see `BroadcastHashJoin` in the plan).
* **E4:** Parquet is smaller on disk, reads only the needed columns, and **partition pruning** skips whole folders.
* **E5:** `coalesce` avoids a shuffle but can give uneven files; `repartition` costs a shuffle but balances the files.
* **E6:** AQE fixes a bad `shuffle.partitions` setting automatically, which is why it is on by default in Spark 3.2+.

## 2. Streaming performance vs requirements

In [ ]:
prog = pd.read_json(config.STREAM_METRICS / "progress.jsonl", lines=True)
busy = prog[prog["input_rows"] > 0]
stream_summary = busy.groupby("query").agg(batches=("batch_id", "count"), rows=("input_rows", "sum"),
                                           avg_rows_per_batch=("input_rows", "mean"),
                                           avg_processed_rows_per_s=("processed_rows_per_sec", "mean"),
                                           p50_batch_ms=("trigger_ms", "median"), p95_batch_ms=("trigger_ms", lambda s: s.quantile(.95))).round(1)
stream_summary

In [ ]:
lat = pd.concat([pd.read_parquet(config.GOLD / "stream" / t, columns=["latency_sec"]) for t in ["traffic_predictions", "air_predictions"]])
p50, p95 = lat["latency_sec"].quantile([.5, .95])
req = pd.DataFrame([
    ["End-to-end latency p50 (sensor file → result written)", "< 10 s", f"{p50:.2f} s", "✅" if p50 < 10 else "❌"],
    ["End-to-end latency p95", "< 10 s", f"{p95:.2f} s", "✅" if p95 < 10 else "❌"],
    ["Micro-batch trigger", "5 s", "5 s (processingTime trigger)", "✅"],
    ["Late-data tolerance", "watermark", "2 h of sensor time", "✅"],
    ["Fault tolerance", "exactly-once output", "checkpoint + replayable source + idempotent batch_id writes", "✅"],
], columns=["Requirement", "Target", "Achieved", ""])
req

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
t = busy[busy["query"] == "traffic_predictions"]
ax.plot(t["batch_id"], t["trigger_ms"], color="#2a78d6", linewidth=2, label="traffic_predictions")
ax.axhline(5000, color="#d03b3b", linestyle="--", linewidth=1, label="5 s trigger interval")
ax.set_xlabel("Micro-batch"); ax.set_ylabel("Batch duration (ms)"); ax.set_title("Each micro-batch finishes well inside the trigger interval")
ax.legend(frameon=False); ax.grid(alpha=.25); ax.spines[["top", "right"]].set_visible(False)
fig.savefig(config.PERF / "streaming_batch_duration.png", dpi=130, bbox_inches="tight"); plt.show()

**Throughput stress test (optional):** run the replayer with `--hours-per-tick 200 --interval 0.5` from `--start 2015-11-08`. That sends about 1,600 events/s. Then re-run this section: `processed_rows_per_s` shows the maximum rate the pipeline can sustain.

## 3. Model quality

In [ ]:
m = json.loads((config.ML_OUT / "metrics.json").read_text())
rows = [{"model": b["model"], "test_RMSE": b["test"]["rmse"], "test_R2": b["test"]["r2"]} for b in m["traffic"]["baselines"]]
rows += [{"model": c["model"], "test_RMSE": c["test"]["rmse"], "test_R2": c["test"]["r2"]} for c in m["traffic"]["candidates"]]
df = pd.DataFrame(rows)
best = m["traffic"]["best"]
base = m["traffic"]["baselines"][0]["test"]["rmse"]
print(f"Selected on validation: {best['model']} → test RMSE {best['test']['rmse']} vs last-hour baseline {base} "
      f"({(1 - best['test']['rmse'] / base) * 100:.1f}% lower error)")
df

> 📸 **Screenshot 8:** the tables and chart in this notebook. Together with Notebook 3 they form the *Performance analysis* section of the report.